# Gold — SLA mensal de entregas por transportadora

Desenvolvido por: Ygor Moraes

## Objetivo

Criar a Gold `gold_ecommerce_rastreamento_entregas_sla_mensal`, medindo o cumprimento de SLA das entregas por mês e transportadora.

## Regra de negócio

Considerar apenas entregas concluídas:

`status_entrega = "entregue"`

Dentro do SLA:

`dias_em_transito <= 7`

Fora do SLA:

`dias_em_transito > 7`

A Gold calcula:

- quantidade de pedidos entregues;
- quantidade de pedidos dentro do SLA;
- quantidade de pedidos fora do SLA;
- percentual dentro do SLA;
- percentual fora do SLA;
- SLA prometido em dias.

## Fonte

- Silver `ecommerce_rastreamento_entregas`

## Cuidados técnicos

- A Silver é lida como Delta.
- Deve ser mantida uma única entrega por pedido.
- A Gold deve manter uma linha por mês e transportadora.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa funções e define parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    countDistinct,
    current_timestamp,
    lit,
    month,
    round as spark_round,
    sum as spark_sum,
    to_date,
    when,
    year,
    row_number
)

from pyspark.sql.window import Window

SILVER_RASTREAMENTO_TABLE = "ecommerce_rastreamento_entregas"

SILVER_RASTREAMENTO_PATH = f"{SILVER_BASE_PATH}{SILVER_RASTREAMENTO_TABLE}"

GOLD_TABLE = "gold_ecommerce_rastreamento_entregas_sla_mensal"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

STATUS_ENTREGA_CONSIDERADO = "entregue"
SLA_PROMETIDO_DIAS = 7

RASTREAMENTO_REQUIRED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "dias_em_transito"
]

GOLD_KEY_COLUMNS = [
    "ano_entrega",
    "mes_entrega",
    "id_transportadora"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_RASTREAMENTO_PATH:", SILVER_RASTREAMENTO_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)
print("SLA_PROMETIDO_DIAS:", SLA_PROMETIDO_DIAS)

In [0]:
# Lê a Silver de rastreamento de entregas.

df_rastreamento = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_RASTREAMENTO_PATH)
)

total_rastreamento = df_rastreamento.count()

print("Silver de rastreamento lida com sucesso.")
print(f"Total eventos rastreamento: {total_rastreamento}")

In [0]:
# Valida colunas obrigatórias, chave e campos principais da fonte.

validate_required_columns(df_rastreamento, RASTREAMENTO_REQUIRED_COLUMNS)

rastreamento_distintos = (
    df_rastreamento
    .select(col("id_rastreamento").cast("int").alias("id_rastreamento"))
    .distinct()
    .count()
)

rastreamento_duplicados = total_rastreamento - rastreamento_distintos

eventos_sem_pedido = (
    df_rastreamento
    .filter(col("id_pedido_ecommerce").isNull())
    .count()
)

eventos_sem_transportadora = (
    df_rastreamento
    .filter(col("id_transportadora").isNull())
    .count()
)

eventos_sem_status = (
    df_rastreamento
    .filter(col("status_entrega").isNull())
    .count()
)

eventos_sem_dt_evento = (
    df_rastreamento
    .filter(col("dt_evento").isNull())
    .count()
)

eventos_sem_dias_em_transito = (
    df_rastreamento
    .filter(col("dias_em_transito").isNull())
    .count()
)

print(f"Total eventos rastreamento: {total_rastreamento}")
print(f"Eventos distintos por id_rastreamento: {rastreamento_distintos}")
print(f"Eventos duplicados por id_rastreamento: {rastreamento_duplicados}")
print(f"Eventos sem id_pedido_ecommerce: {eventos_sem_pedido}")
print(f"Eventos sem id_transportadora: {eventos_sem_transportadora}")
print(f"Eventos sem status_entrega: {eventos_sem_status}")
print(f"Eventos sem dt_evento: {eventos_sem_dt_evento}")
print(f"Eventos sem dias_em_transito: {eventos_sem_dias_em_transito}")

if total_rastreamento == 0:
    raise Exception("Erro: a Silver de rastreamento está vazia.")

if rastreamento_duplicados > 0:
    raise Exception("Erro: existem eventos duplicados por id_rastreamento.")

if eventos_sem_pedido > 0:
    raise Exception("Erro: existem eventos sem id_pedido_ecommerce.")

if eventos_sem_transportadora > 0:
    raise Exception("Erro: existem eventos sem id_transportadora.")

if eventos_sem_status > 0:
    raise Exception("Erro: existem eventos sem status_entrega.")

if eventos_sem_dt_evento > 0:
    raise Exception("Erro: existem eventos sem dt_evento.")

if eventos_sem_dias_em_transito > 0:
    raise Exception("Erro: existem eventos sem dias_em_transito.")

print("Validação OK: fonte mínima conferida.")

In [0]:
# Filtra apenas eventos de entrega concluída.

df_entregas = (
    df_rastreamento
    .filter(col("status_entrega") == STATUS_ENTREGA_CONSIDERADO)
    .select(
        col("id_rastreamento").cast("int").alias("id_rastreamento"),
        col("id_pedido_ecommerce").cast("int").alias("id_pedido_ecommerce"),
        col("id_transportadora").cast("int").alias("id_transportadora"),
        col("dt_evento"),
        col("dias_em_transito").cast("int").alias("dias_em_transito")
    )
    .withColumn("ano_entrega", year(col("dt_evento")))
    .withColumn("mes_entrega", month(col("dt_evento")))
)

total_entregas = df_entregas.count()

print(f"Total eventos com status entregue: {total_entregas}")

if total_entregas == 0:
    raise Exception("Erro: nenhum evento entregue encontrado.")

In [0]:
# Mantém uma única entrega por pedido.

window_entregas = (
    Window
    .partitionBy("id_pedido_ecommerce")
    .orderBy(col("dt_evento").desc_nulls_last())
)

df_entregas_dedup = (
    df_entregas
    .withColumn("rn", row_number().over(window_entregas))
    .filter(col("rn") == 1)
    .drop("rn")
)

total_entregas_dedup = df_entregas_dedup.count()

pedidos_entregues_distintos = (
    df_entregas_dedup
    .select("id_pedido_ecommerce")
    .distinct()
    .count()
)

pedidos_duplicados = total_entregas_dedup - pedidos_entregues_distintos

print(f"Total entregas antes da deduplicação: {total_entregas}")
print(f"Total entregas após deduplicação: {total_entregas_dedup}")
print(f"Pedidos entregues distintos: {pedidos_entregues_distintos}")
print(f"Pedidos duplicados após deduplicação: {pedidos_duplicados}")

if pedidos_duplicados > 0:
    raise Exception("Erro: ainda existem pedidos duplicados após deduplicação.")

print("Validação OK: entregas deduplicadas por pedido.")

In [0]:
# Cria a Gold de SLA mensal por transportadora.

df_gold = (
    df_entregas_dedup
    .withColumn(
        "fl_dentro_sla",
        when(col("dias_em_transito") <= SLA_PROMETIDO_DIAS, 1).otherwise(0)
    )
    .groupBy(
        "ano_entrega",
        "mes_entrega",
        "id_transportadora"
    )
    .agg(
        countDistinct("id_pedido_ecommerce").alias("qtd_pedidos_entregues"),
        spark_sum("fl_dentro_sla").alias("qtd_pedidos_dentro_sla")
    )
    .withColumn(
        "qtd_pedidos_fora_sla",
        col("qtd_pedidos_entregues") - col("qtd_pedidos_dentro_sla")
    )
    .withColumn(
        "percentual_dentro_sla",
        spark_round((col("qtd_pedidos_dentro_sla") / col("qtd_pedidos_entregues")) * 100, 2)
    )
    .withColumn(
        "percentual_fora_sla",
        spark_round((col("qtd_pedidos_fora_sla") / col("qtd_pedidos_entregues")) * 100, 2)
    )
    .withColumn("sla_prometido_dias", lit(SLA_PROMETIDO_DIAS))
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("ano_entrega", "mes_entrega", "id_transportadora")
)

print("Gold de SLA mensal criada em memória.")
display(df_gold)

In [0]:
# Valida totais, chaves e campos principais da Gold.

total_linhas_gold = df_gold.count()

total_chaves_gold = (
    df_gold
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_gold = total_linhas_gold - total_chaves_gold

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues"),
        spark_sum("qtd_pedidos_dentro_sla").alias("total_pedidos_dentro_sla"),
        spark_sum("qtd_pedidos_fora_sla").alias("total_pedidos_fora_sla")
    )
    .collect()[0]
)

nulos_gold = (
    df_gold
    .filter(
        col("ano_entrega").isNull() |
        col("mes_entrega").isNull() |
        col("id_transportadora").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("qtd_pedidos_dentro_sla").isNull() |
        col("qtd_pedidos_fora_sla").isNull() |
        col("percentual_dentro_sla").isNull() |
        col("percentual_fora_sla").isNull() |
        col("sla_prometido_dias").isNull()
    )
    .count()
)

print(f"Total pedidos entregues deduplicados: {total_entregas_dedup}")
print(f"Total pedidos entregues na Gold: {validacao_gold['total_pedidos_entregues']}")
print(f"Pedidos dentro SLA na Gold: {validacao_gold['total_pedidos_dentro_sla']}")
print(f"Pedidos fora SLA na Gold: {validacao_gold['total_pedidos_fora_sla']}")
print(f"Total linhas Gold: {total_linhas_gold}")
print(f"Chaves duplicadas Gold: {chaves_duplicadas_gold}")
print(f"Linhas com nulos principais: {nulos_gold}")

if validacao_gold["total_pedidos_entregues"] != total_entregas_dedup:
    raise Exception("Erro: total de pedidos entregues da Gold não fecha com a base.")

if (
    validacao_gold["total_pedidos_dentro_sla"] +
    validacao_gold["total_pedidos_fora_sla"]
    != validacao_gold["total_pedidos_entregues"]
):
    raise Exception("Erro: pedidos dentro + fora SLA não fecha com total.")

if chaves_duplicadas_gold > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold.")

if nulos_gold > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold.")

print("Validação OK: Gold em memória conferida.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .partitionBy("ano_entrega", "mes_entrega")
    .save(GOLD_PATH)
)

print(f"Gold gravada com sucesso em Delta: {GOLD_PATH}")

In [0]:
# Lê e valida a Gold Delta gravada.

df_gold_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

total_linhas_gold_saved = df_gold_saved.count()

total_chaves_gold_saved = (
    df_gold_saved
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_gold_saved = total_linhas_gold_saved - total_chaves_gold_saved

validacao_gold_saved = (
    df_gold_saved
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues"),
        spark_sum("qtd_pedidos_dentro_sla").alias("total_pedidos_dentro_sla"),
        spark_sum("qtd_pedidos_fora_sla").alias("total_pedidos_fora_sla")
    )
    .collect()[0]
)

nulos_gold_saved = (
    df_gold_saved
    .filter(
        col("ano_entrega").isNull() |
        col("mes_entrega").isNull() |
        col("id_transportadora").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("qtd_pedidos_dentro_sla").isNull() |
        col("qtd_pedidos_fora_sla").isNull() |
        col("percentual_dentro_sla").isNull() |
        col("percentual_fora_sla").isNull() |
        col("sla_prometido_dias").isNull()
    )
    .count()
)

print(f"Total linhas Gold Delta: {total_linhas_gold_saved}")
print(f"Chaves duplicadas Gold Delta: {chaves_duplicadas_gold_saved}")
print(f"Total pedidos entregues base: {total_entregas_dedup}")
print(f"Total pedidos entregues Gold Delta: {validacao_gold_saved['total_pedidos_entregues']}")
print(f"Pedidos dentro SLA Gold Delta: {validacao_gold_saved['total_pedidos_dentro_sla']}")
print(f"Pedidos fora SLA Gold Delta: {validacao_gold_saved['total_pedidos_fora_sla']}")
print(f"Linhas com nulos principais Gold Delta: {nulos_gold_saved}")

if validacao_gold_saved["total_pedidos_entregues"] != total_entregas_dedup:
    raise Exception("Erro: total de pedidos entregues da Gold Delta não confere.")

if (
    validacao_gold_saved["total_pedidos_dentro_sla"] +
    validacao_gold_saved["total_pedidos_fora_sla"]
    != validacao_gold_saved["total_pedidos_entregues"]
):
    raise Exception("Erro: pedidos dentro + fora SLA não fecha na Gold Delta.")

if chaves_duplicadas_gold_saved > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold Delta.")

if nulos_gold_saved > 0:
    raise Exception("Erro: existem nulos nas colunas principais da Gold Delta.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_saved
    .select(
        col("ano_entrega").cast("int").alias("ano_entrega"),
        col("mes_entrega").cast("int").alias("mes_entrega"),
        col("id_transportadora").cast("int").alias("id_transportadora"),
        col("qtd_pedidos_entregues").cast("int").alias("qtd_pedidos_entregues"),
        col("qtd_pedidos_dentro_sla").cast("int").alias("qtd_pedidos_dentro_sla"),
        col("qtd_pedidos_fora_sla").cast("int").alias("qtd_pedidos_fora_sla"),
        col("percentual_dentro_sla").cast("decimal(10,2)").alias("percentual_dentro_sla"),
        col("percentual_fora_sla").cast("decimal(10,2)").alias("percentual_fora_sla"),
        col("sla_prometido_dias").cast("int").alias("sla_prometido_dias"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")
df_gold_sql.printSchema()
display(df_gold_sql.orderBy("ano_entrega", "mes_entrega", "id_transportadora"))

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final do SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_linhas_final = df_final.count()

total_chaves_final = (
    df_final
    .select(GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

chaves_duplicadas_final = total_linhas_final - total_chaves_final

validacao_final = (
    df_final
    .agg(
        spark_sum("qtd_pedidos_entregues").alias("total_pedidos_entregues"),
        spark_sum("qtd_pedidos_dentro_sla").alias("total_pedidos_dentro_sla"),
        spark_sum("qtd_pedidos_fora_sla").alias("total_pedidos_fora_sla")
    )
    .collect()[0]
)

nulos_final = (
    df_final
    .filter(
        col("ano_entrega").isNull() |
        col("mes_entrega").isNull() |
        col("id_transportadora").isNull() |
        col("qtd_pedidos_entregues").isNull() |
        col("qtd_pedidos_dentro_sla").isNull() |
        col("qtd_pedidos_fora_sla").isNull() |
        col("percentual_dentro_sla").isNull() |
        col("percentual_fora_sla").isNull() |
        col("sla_prometido_dias").isNull()
    )
    .count()
)

print(f"Total linhas tabela final: {total_linhas_final}")
print(f"Chaves duplicadas tabela final: {chaves_duplicadas_final}")
print(f"Total pedidos entregues base: {total_entregas_dedup}")
print(f"Total pedidos entregues tabela final: {validacao_final['total_pedidos_entregues']}")
print(f"Pedidos dentro SLA tabela final: {validacao_final['total_pedidos_dentro_sla']}")
print(f"Pedidos fora SLA tabela final: {validacao_final['total_pedidos_fora_sla']}")
print(f"Linhas com nulos principais tabela final: {nulos_final}")

if validacao_final["total_pedidos_entregues"] != total_entregas_dedup:
    raise Exception("Erro: total de pedidos entregues da tabela final não confere.")

if (
    validacao_final["total_pedidos_dentro_sla"] +
    validacao_final["total_pedidos_fora_sla"]
    != validacao_final["total_pedidos_entregues"]
):
    raise Exception("Erro: pedidos dentro + fora SLA não fecha na tabela final.")

if chaves_duplicadas_final > 0:
    raise Exception("Erro: existem chaves duplicadas na tabela final.")

if nulos_final > 0:
    raise Exception("Erro: existem nulos nas colunas principais da tabela final.")

print("Validação OK: tabela final SQL Server gravada corretamente.")